In [ ]:
# ============================================================
# CELL 0 — COLAB SETUP
# Run this FIRST at the start of every new Colab session.
# Takes ~1 min. Safe to re-run.
# ============================================================

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 2. Install packages not pre-installed on Colab
!pip install praat-parselmouth noisereduce soundfile transformers torchaudio openai-whisper -q

# 3. Create all project directories on Drive (idempotent)
import os

PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"

dirs = [
    "outputs",
    "data/train/native",
    "data/train/non_native",
    "data/preprocessed/native",
    "data/preprocessed/non_native",
    "data/embeddings",
    "data/segments/native",
    "data/segments/non_native",
    "data/augmented/native",
    "data/augmented/non_native",
]
for d in dirs:
    os.makedirs(f"{PROJECT_ROOT}/{d}", exist_ok=True)

# 4. Verify GPU (critical for Stage 6A)
import torch
print(f"\nGPU available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"VRAM          : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️  No GPU — Runtime → Change runtime type → T4 GPU")
    print("   Stage 6A will be very slow on CPU.")

# 5. Check renan_dataset.csv is on Drive
csv_path = f"{PROJECT_ROOT}/renan_dataset.csv"
if os.path.exists(csv_path):
    import pandas as pd
    df = pd.read_csv(csv_path)
    print(f"\n✅ renan_dataset.csv found ({len(df)} rows)")
else:
    print(f"\n❌ renan_dataset.csv NOT found.")
    print(f"   Upload it manually to: {PROJECT_ROOT}/")
    print(f"   (Google Drive → team_databaes folder → Upload file)")

print(f"\nProject root  : {PROJECT_ROOT}")
print(f"Setup complete ✅")

In [ ]:
# ============================================================
# CELL 1 — STAGE 1: DATA EXPLORATION
# Reads renan_dataset.csv from Drive.
# All outputs saved directly to Drive.
# Safe to re-run — overwrites previous outputs.
# ============================================================

import os, io, time, requests, librosa, numpy as np, pandas as pd
from tqdm.notebook import tqdm

PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"
EXCEL_PATH   = f"{PROJECT_ROOT}/renan_dataset.csv"
OUTPUT_DIR   = f"{PROJECT_ROOT}/outputs"
TARGET_SR    = 16000

# Set to None to explore all 160 files (takes ~5 min)
# Keep at 50 for a quick sanity check (~1 min)
EXPLORE_N = None

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── SKIP IF ALREADY DONE ─────────────────────────────────────
STAGE1_OUT = f"{OUTPUT_DIR}/stage1_exploration.csv"
if os.path.exists(STAGE1_OUT):
    print(f"⏭️  Stage 1 already complete — loading existing results.")
    df_stats = pd.read_csv(STAGE1_OUT)
    df_raw   = pd.read_csv(EXCEL_PATH)
    df_raw["label"] = df_raw["nativity_status"].map({"Native": 1, "Non-Native": 0})
    print(f"   Loaded {len(df_stats)} rows from {STAGE1_OUT}")
    print("   Delete outputs/stage1_exploration.csv to force re-run.")
    print("\n✅ Stage 1 already done — skipped.")
    raise SystemExit(0)

# ── Load spreadsheet ──────────────────────────────────────────
print("📂 Loading dataset spreadsheet...")
df_raw = pd.read_csv(EXCEL_PATH)
print(f"   Total rows loaded: {len(df_raw)}")
print(f"   Columns: {list(df_raw.columns)}")

df_raw["label"] = df_raw["nativity_status"].map({"Native": 1, "Non-Native": 0})

unknown_labels = df_raw[df_raw["label"].isna()]
if len(unknown_labels) > 0:
    print(f"⚠️  {len(unknown_labels)} rows with unrecognised nativity_status:")
    print(unknown_labels["nativity_status"].unique())

df_explore = df_raw.sample(n=EXPLORE_N, random_state=42) if EXPLORE_N else df_raw
print(f"\n🔍 Exploring {len(df_explore)} files...\n")

# ── Download + inspect ────────────────────────────────────────
def download_and_inspect(row):
    dp_id   = row["dp_id"]
    url     = row["audio_url"]
    label   = row["label"]
    dialect = row["language"]

    result = {
        "dp_id":    dp_id, "label": label,
        "nativity": row["nativity_status"], "dialect": dialect, "url": url,
    }
    try:
        response    = requests.get(url, timeout=15)
        response.raise_for_status()
        audio_bytes = io.BytesIO(response.content)

        y, sr = librosa.load(audio_bytes, sr=None, mono=True)
        duration      = librosa.get_duration(y=y, sr=sr)
        rms           = float(np.sqrt(np.mean(y ** 2)))
        zcr           = float(np.mean(librosa.feature.zero_crossing_rate(y)))
        frame_rms     = librosa.feature.rms(y=y)[0]
        silence_ratio = float(np.mean(frame_rms < 0.01 * frame_rms.max()))

        result.update({
            "original_sr":    sr,
            "duration_s":     round(duration, 2),
            "rms_volume":     round(rms, 5),
            "zero_cross_rate":round(zcr, 5),
            "silence_ratio":  round(silence_ratio, 3),
            "n_samples":      len(y),
            "usable":         duration >= 3.0,
            "error":          None,
        })
    except requests.exceptions.Timeout:
        result.update({"error": "TIMEOUT", "usable": False})
    except requests.exceptions.HTTPError as e:
        result.update({"error": f"HTTP_{e.response.status_code}", "usable": False})
    except Exception as e:
        result.update({"error": str(e)[:80], "usable": False})
    return result

records = []
for _, row in tqdm(df_explore.iterrows(), total=len(df_explore), desc="Inspecting audio"):
    records.append(download_and_inspect(row))
    time.sleep(0.05)

df_stats = pd.DataFrame(records)

# ── Print report ──────────────────────────────────────────────
sep = "=" * 58
report_lines = []

def log(line=""):
    print(line)
    report_lines.append(line)

log(sep)
log("  STAGE 1 — DATA EXPLORATION REPORT")
log(f"  Dataset: renan_dataset.csv  |  Files inspected: {len(df_stats)}")
log(sep)

log("\n📊 CLASS BALANCE")
log("-" * 35)
balance = df_raw["nativity_status"].value_counts()
for name, count in balance.items():
    pct = 100 * count / len(df_raw)
    log(f"   {name:<15}: {count:>5} files  ({pct:.1f}%)")
imbalance_ratio = balance.max() / balance.min()
log(f"\n   Imbalance ratio: {imbalance_ratio:.1f}x")
if imbalance_ratio > 2:
    log("   ⚠️  Significant imbalance — class_weight='balanced' is essential in Stage 8")

log("\n🌍 DIALECT DISTRIBUTION")
log("-" * 35)
for dialect, count in df_raw["language"].value_counts().items():
    pct = 100 * count / len(df_raw)
    log(f"   {dialect:<18}: {count:>4} files  ({pct:.1f}%)")

log("\n⏱️  DURATION STATS (seconds)")
log("-" * 35)
valid = df_stats[df_stats["error"].isna()]
for label_val, label_name in [(1, "Native"), (0, "Non-Native")]:
    subset = valid[valid["label"] == label_val]["duration_s"]
    if len(subset) == 0:
        continue
    log(f"\n   {label_name}:")
    log(f"     mean={subset.mean():.1f}s  median={subset.median():.1f}s  "
        f"min={subset.min():.1f}s  max={subset.max():.1f}s")
    est_segs = subset.apply(lambda d: max(0, int((d - 3) / 1) + 1))
    log(f"     Estimated 3s segments per file: ~{est_segs.mean():.0f} avg, "
        f"{est_segs.sum()} total")

log("\n🎵 SAMPLE RATES FOUND (original, before resampling)")
log("-" * 35)
for sr, count in valid["original_sr"].value_counts().items():
    flag = "✅" if sr == TARGET_SR else "⚠️  will resample"
    log(f"   {sr} Hz : {count} files  {flag}")

log("\n🔊 VOLUME (RMS) BY CLASS")
log("-" * 35)
rms_stats = valid.groupby("label")["rms_volume"].describe()
log(rms_stats.round(5).to_string())
rms_native    = valid[valid["label"]==1]["rms_volume"].mean()
rms_nonnative = valid[valid["label"]==0]["rms_volume"].mean()
if abs(rms_native - rms_nonnative) / max(rms_native, rms_nonnative) > 0.2:
    log("   ⚠️  Noticeable volume difference — RMS normalization in Stage 3 is important")

log("\n🤫 SILENCE RATIO BY CLASS  (0=no silence, 1=all silence)")
log("-" * 35)
log(valid.groupby("label")["silence_ratio"].describe().round(3).to_string())

log("\n⚠️  PROBLEMATIC FILES")
log("-" * 35)
errors = df_stats[df_stats["error"].notna()]
short  = df_stats[(df_stats["error"].isna()) & (df_stats["duration_s"] < 3.0)]
log(f"   Download/load errors : {len(errors)}")
log(f"   Too short (<3s)      : {len(short)}  ← will be dropped in Stage 3")
log(f"   Usable files         : {df_stats['usable'].sum()} / {len(df_stats)}")
if len(errors) > 0:
    log("\n   Error breakdown:")
    log(errors["error"].value_counts().to_string())

log("\n" + sep)
log("  KEY TAKEAWAYS FOR DOWNSTREAM STAGES")
log(sep)
log(f"  Stage 3: Resample all audio to {TARGET_SR}Hz mono + RMS normalize")
log(f"  Stage 4: Use 3s window / 1s hop for segmentation")
log(f"  Stage 5: Augment training data only (non-native is minority class)")
log(f"  Stage 8: Use class_weight='balanced' in RF due to imbalance")
log(sep)

# ── Save outputs directly to Drive ───────────────────────────
df_stats.to_csv(f"{OUTPUT_DIR}/stage1_exploration.csv", index=False)
log(f"\n💾 Saved: {OUTPUT_DIR}/stage1_exploration.csv")

with open(f"{OUTPUT_DIR}/stage1_report.txt", "w") as f:
    f.write("\n".join(report_lines))
log(f"💾 Saved: {OUTPUT_DIR}/stage1_report.txt")

print("\n✅ Stage 1 complete.")

In [ ]:
# ============================================================
# CELL 2 — STAGE 2: DATA COLLECTION
# Part A: Download all 160 Renan MP3s → Drive
# Part B: Fetch 200 CV native clips via Mozilla API (streaming)
#         ✅ NO full 3.25GB download — streams tar and extracts
#            only the ~200 clips we need (~10MB total written)
# Part C: Build master manifest → Drive
#
# ── BEFORE RUNNING ──────────────────────────────────────────
# 1. Get your API key from:
#    https://datacollective.mozillafoundation.org
#    (Account → API Keys → Create)
# 2. Paste it into CV_API_KEY below
# ────────────────────────────────────────────────────────────
# Safe to re-run — skips already-downloaded files.
# ============================================================

import os, io, time, tarfile, hashlib, requests
import numpy as np, pandas as pd
from tqdm.notebook import tqdm

PROJECT_ROOT    = "/content/drive/MyDrive/team_databaes"
RENAN_CSV       = f"{PROJECT_ROOT}/renan_dataset.csv"
TRAIN_NATIVE    = f"{PROJECT_ROOT}/data/train/native"
TRAIN_NONNATIVE = f"{PROJECT_ROOT}/data/train/non_native"
OUTPUT_DIR      = f"{PROJECT_ROOT}/outputs"

# ── ⚠️  PASTE YOUR API KEY HERE ──────────────────────────────
CV_API_KEY = "YOUR_API_KEY_HERE"
# ─────────────────────────────────────────────────────────────

CV_DATASET_ID  = "cmj8u3os6000tnxxb169x1zdc"
CV_API_URL     = f"https://datacollective.mozillafoundation.org/api/datasets/{CV_DATASET_ID}/download"
CV_NATIVE_N    = 200
CV_MIN_UPVOTES = 2
TIMEOUT        = 20
SLEEP          = 0.1

# ════════════════════════════════════════════════════════════
# PART A — DOWNLOAD RENAN FILES
# ════════════════════════════════════════════════════════════
print("=" * 58)
print("  PART A — Downloading Renan Training Audio (160 files)")
print("=" * 58)

df_renan = pd.read_csv(RENAN_CSV)
df_renan["label"]     = df_renan["nativity_status"].map({"Native": 1, "Non-Native": 0})
df_renan["filename"]  = df_renan["dp_id"].apply(lambda x: f"dp_{x}.mp3")
df_renan["save_dir"]  = df_renan["label"].map({1: TRAIN_NATIVE, 0: TRAIN_NONNATIVE})
df_renan["save_path"] = df_renan.apply(
    lambda r: os.path.join(r["save_dir"], r["filename"]), axis=1
)

manifest_records = []

def download_renan_file(url, save_path, dp_id):
    if os.path.exists(save_path):
        return {
            "dp_id": dp_id, "status": "already_exists",
            "file_size_kb": round(os.path.getsize(save_path) / 1024, 1),
            "md5": None, "error": None
        }
    try:
        r = requests.get(url, timeout=TIMEOUT)
        r.raise_for_status()
        ctype = r.headers.get("Content-Type", "")
        if "audio" not in ctype and "octet-stream" not in ctype:
            return {
                "dp_id": dp_id, "status": "bad_content_type",
                "file_size_kb": 0, "md5": None, "error": f"Content-Type={ctype}"
            }
        with open(save_path, "wb") as f:
            f.write(r.content)
        return {
            "dp_id": dp_id, "status": "downloaded",
            "file_size_kb": round(len(r.content) / 1024, 1),
            "md5": hashlib.md5(r.content).hexdigest()[:8], "error": None
        }
    except requests.exceptions.Timeout:
        return {"dp_id": dp_id, "status": "error", "file_size_kb": 0, "md5": None, "error": "TIMEOUT"}
    except requests.exceptions.HTTPError as e:
        return {"dp_id": dp_id, "status": "error", "file_size_kb": 0, "md5": None,
                "error": f"HTTP_{e.response.status_code}"}
    except Exception as e:
        return {"dp_id": dp_id, "status": "error", "file_size_kb": 0, "md5": None,
                "error": str(e)[:80]}

for _, row in tqdm(df_renan.iterrows(), total=len(df_renan), desc="Renan files"):
    result = download_renan_file(row["audio_url"], row["save_path"], row["dp_id"])
    result.update({
        "source":    "renan",
        "label":     row["label"],
        "nativity":  row["nativity_status"],
        "dialect":   row["language"],
        "save_path": row["save_path"],
    })
    manifest_records.append(result)
    time.sleep(SLEEP)

a_df = pd.DataFrame(manifest_records)
print(f"\nRenan download complete:")
print(f"   Downloaded     : {(a_df['status']=='downloaded').sum()}")
print(f"   Already existed: {(a_df['status']=='already_exists').sum()}")
print(f"   Failed         : {(a_df['status'].isin(['error','bad_content_type'])).sum()}")


# ════════════════════════════════════════════════════════════
# PART B — MOZILLA COMMON VOICE 24.0 via API (streaming)
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 58)
print("  PART B — Common Voice 24.0 Arabic via API (streaming)")
print("=" * 58)

cv_records = []

if CV_API_KEY == "YOUR_API_KEY_HERE":
    print("⚠️  CV_API_KEY not set — skipping Part B.")
    print("   Set your API key at the top of this cell and re-run.")
    print("   Continuing with Renan-only data for now.")
else:
    # ── Step 1: Get presigned download URL from API ───────────
    print("Requesting presigned download URL from Mozilla API...")
    try:
        resp = requests.post(
            CV_API_URL,
            headers={
                "Authorization": f"Bearer {CV_API_KEY}",
                "Content-Type":  "application/json",
            },
            timeout=30,
        )
        resp.raise_for_status()
        download_url = resp.json()["downloadUrl"]
        print(f"✅ Got presigned URL (valid for ~15 min)")
    except requests.exceptions.HTTPError as e:
        print(f"❌ API error: HTTP {e.response.status_code}")
        print(f"   Response: {e.response.text[:300]}")
        print("   Check your API key and try again.")
        download_url = None
    except KeyError:
        print(f"❌ Unexpected API response — 'downloadUrl' not in response.")
        print(f"   Full response: {resp.json()}")
        download_url = None
    except Exception as e:
        print(f"❌ Request failed: {e}")
        download_url = None

    if download_url:
        # ── Step 2: PASS 1 — Stream tar to get train.tsv only ──
        # We stream the entire tar.gz but only materialise the TSV.
        # This reads sequentially until we hit the TSV member then stops.
        # Memory usage: only the TSV content (~few MB).
        print("\nPass 1: Streaming archive to extract train.tsv...")
        print("(Reads sequentially until TSV is found — stops early)")

        class StreamingTarWrapper(io.RawIOBase):
            """
            Wraps a requests streaming response so tarfile can read it
            as a file-like object without buffering the entire download.
            """
            def __init__(self, response):
                self._iter    = response.iter_content(chunk_size=1024 * 256)  # 256KB chunks
                self._buf     = b""
                self._pos     = 0
                self.bytes_read = 0

            def readable(self):
                return True

            def readinto(self, b):
                while len(self._buf) < len(b):
                    try:
                        chunk = next(self._iter)
                        self._buf += chunk
                        self.bytes_read += len(chunk)
                    except StopIteration:
                        break
                n = min(len(b), len(self._buf))
                b[:n] = self._buf[:n]
                self._buf = self._buf[n:]
                return n

        # --- Pass 1: extract train.tsv ---
        df_filtered  = None
        clips_prefix = None

        stream_resp1 = requests.get(download_url, stream=True, timeout=120)
        stream_resp1.raise_for_status()
        wrapper1 = StreamingTarWrapper(stream_resp1)

        with tarfile.open(fileobj=wrapper1, mode="r|gz") as tar:
            for member in tar:
                # Grab clips prefix ONLY from Arabic clips (/ar/clips/)
                # Do NOT use first .mp3 found — that is likely a non-Arabic locale
                if (clips_prefix is None and
                        member.name.endswith(".mp3") and
                        "/ar/clips/" in member.name):
                    clips_prefix = member.name.rsplit("/ar/clips/", 1)[0] + "/ar/clips/"
                    print(f"  Detected clips prefix: {clips_prefix}")

                if member.name.endswith("ar/train.tsv"):
                    print(f"  Found TSV: {member.name} ({member.size / 1e6:.1f} MB)")
                    f_obj = tar.extractfile(member)
                    # pd.read_csv needs a seekable fileobj — streaming tar members
                    # are not seekable, so we read the bytes fully into a BytesIO
                    # buffer first. TSV is ~9MB so this is fine in memory.
                    tsv_bytes = io.BytesIO(f_obj.read())
                    df_cv = pd.read_csv(tsv_bytes, sep="\t")
                    print(f"  Train clips in TSV: {len(df_cv):,}")
                    print(f"  Columns: {list(df_cv.columns)}")

                    # Quality filter
                    df_cv["up_votes"]   = pd.to_numeric(df_cv["up_votes"],   errors="coerce").fillna(0)
                    df_cv["down_votes"] = pd.to_numeric(df_cv["down_votes"], errors="coerce").fillna(0)
                    df_filtered = df_cv[
                        (df_cv["up_votes"]   >= CV_MIN_UPVOTES) &
                        (df_cv["down_votes"] == 0)
                    ].copy()
                    print(f"  After quality filter: {len(df_filtered):,}")

                    # CV 24.0 uses 'accents' (plural) not 'accent' (singular)
                    # Detect whichever is present
                    if "accents" in df_filtered.columns:
                        accent_col = "accents"
                    elif "accent" in df_filtered.columns:
                        accent_col = "accent"
                    else:
                        # No accent column at all — treat everything as unspecified
                        accent_col = None
                        df_filtered["_accent"] = "unspecified"
                        accent_col = "_accent"

                    print(f"  Using accent column: '{accent_col}'")

                    # Accent-stratified sampling
                    df_filtered["accent"] = (
                        df_filtered[accent_col].fillna("unspecified").replace("", "unspecified")
                    )
                    accent_groups = df_filtered["accent"].unique()
                    n_groups      = len(accent_groups)
                    per_group     = max(1, CV_NATIVE_N // n_groups)
                    remainder     = CV_NATIVE_N - (per_group * n_groups)

                    sampled_parts = []
                    for i, accent in enumerate(accent_groups):
                        grp    = df_filtered[df_filtered["accent"] == accent]
                        n_take = min(per_group + (1 if i < remainder else 0), len(grp))
                        sampled_parts.append(grp.sample(n=n_take, random_state=42))

                    df_sample = pd.concat(sampled_parts, ignore_index=True)
                    print(f"\n  Accent breakdown:")
                    print(df_filtered["accent"].value_counts().to_string())
                    print(f"\n  Sampled {len(df_sample)} clips across {n_groups} accent groups")
                    break   # ← stop streaming as soon as we have the TSV

        stream_resp1.close()
        print(f"  Pass 1 complete — read {wrapper1.bytes_read / 1e6:.0f} MB of archive")

        if df_filtered is None:
            print("❌ Could not find ar/train.tsv in archive. Skipping CV.")
        else:
            # Build set of clip paths we need
            accent_to_dialect = {
                "saudi":       "Arabic_SA",
                "gulf":        "Arabic_QA",
                "egyptian":    "Arabic_EG",
                "levantine":   "Arabic_LEV",
                "moroccan":    "Arabic_MA",
                "unspecified": "Arabic_MSA",
            }

            # Build a lookup: tar path → (out_path, dialect, accent, idx, up_votes)
            clips_needed = {}
            for idx, row in df_sample.iterrows():
                accent_slug      = str(row["accent"]).replace(" ", "_").lower()
                out_fname        = f"cv_{accent_slug}_{row['path']}"
                out_path         = os.path.join(TRAIN_NATIVE, out_fname)
                clip_path_in_tar = (clips_prefix or "cv-corpus-24.0-2025-12-05/ar/clips/") + row["path"]
                dialect_label    = accent_to_dialect.get(str(row["accent"]).lower(), "Arabic_MSA")

                clips_needed[clip_path_in_tar] = {
                    "dp_id":     f"cv_{idx:06d}",
                    "out_path":  out_path,
                    "dialect":   dialect_label,
                    "cv_accent": row["accent"],
                    "up_votes":  int(row["up_votes"]),
                }

            # Debug: print a sample of the expected tar paths so we can verify
            sample_keys = list(clips_needed.keys())[:3]
            print(f"  Sample expected tar paths:")
            for k in sample_keys:
                print(f"    {k}")

            # Skip clips already on Drive
            already_on_disk = {
                k: v for k, v in clips_needed.items()
                if os.path.exists(v["out_path"])
            }
            to_fetch = {
                k: v for k, v in clips_needed.items()
                if not os.path.exists(v["out_path"])
            }

            print(f"\nClips already on Drive : {len(already_on_disk)}")
            print(f"Clips to fetch         : {len(to_fetch)}")

            # Add already-on-disk clips to cv_records immediately
            for tar_path, info in already_on_disk.items():
                cv_records.append({
                    "dp_id":     info["dp_id"],
                    "source":    "mozilla_common_voice_24",
                    "label":     1,
                    "nativity":  "Native",
                    "dialect":   info["dialect"],
                    "save_path": info["out_path"],
                    "status":    "already_exists",
                    "error":     None,
                    "cv_accent": info["cv_accent"],
                    "up_votes":  info["up_votes"],
                })

            if to_fetch:
                # ── Step 3: PASS 2 — Stream again, save only needed clips ──
                # We stream the full archive but only write out the ~200
                # target clips. Everything else is discarded in memory.
                # Total written to disk: ~200 clips × ~50KB = ~10MB.
                print(f"\nPass 2: Streaming archive to extract {len(to_fetch)} clips...")
                print("(Reads full archive sequentially — ~3.25GB streamed, ~10MB written)")

                # Need a fresh presigned URL if the old one might have expired
                # (Pass 1 may have taken >15 min for large archives)
                print("Refreshing presigned URL for Pass 2...")
                try:
                    resp2 = requests.post(
                        CV_API_URL,
                        headers={
                            "Authorization": f"Bearer {CV_API_KEY}",
                            "Content-Type":  "application/json",
                        },
                        timeout=30,
                    )
                    resp2.raise_for_status()
                    download_url2 = resp2.json()["downloadUrl"]
                    print("✅ Got fresh presigned URL")
                except Exception as e:
                    print(f"⚠️  Could not refresh URL: {e} — trying original URL")
                    download_url2 = download_url

                stream_resp2 = requests.get(download_url2, stream=True, timeout=120)
                stream_resp2.raise_for_status()
                wrapper2 = StreamingTarWrapper(stream_resp2)

                extracted = 0
                errors    = 0
                pbar      = tqdm(total=len(to_fetch), desc="Extracting CV clips")

                with tarfile.open(fileobj=wrapper2, mode="r|gz") as tar:
                    for member in tar:
                        if member.name not in to_fetch:
                            # Skip — but we must still read past it in the stream
                            # tarfile does this automatically for r| (streaming) mode
                            continue

                        info  = to_fetch[member.name]
                        f_obj = tar.extractfile(member)

                        if f_obj is None:
                            cv_records.append({
                                "dp_id":     info["dp_id"],
                                "source":    "mozilla_common_voice_24",
                                "label":     1,
                                "nativity":  "Native",
                                "dialect":   info["dialect"],
                                "save_path": None,
                                "status":    "error",
                                "error":     "Empty tar member",
                                "cv_accent": info["cv_accent"],
                                "up_votes":  info["up_votes"],
                            })
                            errors += 1
                        else:
                            try:
                                with open(info["out_path"], "wb") as out_f:
                                    out_f.write(f_obj.read())
                                cv_records.append({
                                    "dp_id":     info["dp_id"],
                                    "source":    "mozilla_common_voice_24",
                                    "label":     1,
                                    "nativity":  "Native",
                                    "dialect":   info["dialect"],
                                    "save_path": info["out_path"],
                                    "status":    "extracted",
                                    "error":     None,
                                    "cv_accent": info["cv_accent"],
                                    "up_votes":  info["up_votes"],
                                })
                                extracted += 1
                            except Exception as e:
                                cv_records.append({
                                    "dp_id":     info["dp_id"],
                                    "source":    "mozilla_common_voice_24",
                                    "label":     1,
                                    "nativity":  "Native",
                                    "dialect":   info["dialect"],
                                    "save_path": None,
                                    "status":    "error",
                                    "error":     str(e)[:80],
                                    "cv_accent": info["cv_accent"],
                                    "up_votes":  info["up_votes"],
                                })
                                errors += 1

                        pbar.update(1)
                        # Stop early once we have everything
                        if extracted + errors + len(already_on_disk) >= len(clips_needed):
                            break

                pbar.close()
                stream_resp2.close()
                print(f"\nPass 2 complete — read {wrapper2.bytes_read / 1e6:.0f} MB of archive")

                print(f"\nCommon Voice extraction complete:")
                print(f"   Extracted       : {extracted}")
                print(f"   Already existed : {len(already_on_disk)}")
                print(f"   Errors          : {errors}")
                if cv_records:
                    b_df = pd.DataFrame(cv_records)
                    print(f"   Accent breakdown:")
                    print(b_df["cv_accent"].value_counts().to_string())
                else:
                    print("   WARNING: No clips recorded — check clips_prefix above")

    manifest_records.extend(cv_records)


# ════════════════════════════════════════════════════════════
# PART C — MASTER MANIFEST
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 58)
print("  PART C — Master Manifest")
print("=" * 58)

manifest_df = pd.DataFrame(manifest_records)

usable = manifest_df[
    manifest_df["status"].isin(["downloaded", "already_exists", "extracted"]) &
    manifest_df["save_path"].notna()
].copy()

usable["file_on_disk"] = usable["save_path"].apply(os.path.exists)
ghost = (~usable["file_on_disk"]).sum()
if ghost:
    print(f"Removing {ghost} manifest entries where file missing on disk")
usable = usable[usable["file_on_disk"]].drop(columns=["file_on_disk"])

n_native    = (usable["label"] == 1).sum()
n_nonnative = (usable["label"] == 0).sum()
ratio       = n_native / n_nonnative if n_nonnative > 0 else float("inf")

print(f"\nFinal manifest:")
print(f"   Total usable files : {len(usable)}")
print(f"\n   By source:")
print(usable["source"].value_counts().to_string())
print(f"\n   By class:")
print(usable["nativity"].value_counts().to_string())
print(f"\n   By dialect:")
print(usable["dialect"].value_counts().to_string())
print(f"\n   Class balance:")
print(f"     Native     : {n_native}")
print(f"     Non-Native : {n_nonnative}")
print(f"     Ratio      : {ratio:.1f}x")

manifest_df.to_csv(f"{OUTPUT_DIR}/stage2_manifest.csv", index=False)
print(f"\nSaved: {OUTPUT_DIR}/stage2_manifest.csv")
print("\n✅ Stage 2 complete.")
print("⏭️  Next: Run CELL_3_stage3.py")


In [ ]:
# ============================================================
# CELL 3 — STAGE 3: PREPROCESSING
# Reads stage2_manifest.csv from Drive.
# Preprocessed WAVs saved directly to Drive.
#
# Pipeline per file: resample → RMS normalize → VAD trim
#                    → denoise → duration gate → save WAV
#
# RESUME SAFE: skips files whose output WAV already exists on
# Drive. If the session drops mid-run, re-run this cell and
# it will continue from where it stopped.
# ============================================================

import os, warnings
import numpy as np, pandas as pd
import librosa, soundfile as sf, noisereduce as nr
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

PROJECT_ROOT  = "/content/drive/MyDrive/team_databaes"
MANIFEST_IN   = f"{PROJECT_ROOT}/outputs/stage2_manifest.csv"
MANIFEST_OUT  = f"{PROJECT_ROOT}/outputs/stage3_manifest.csv"
PREPROC_DIR   = f"{PROJECT_ROOT}/data/preprocessed"

TARGET_SR    = 16000
TARGET_RMS   = 0.05
MIN_DURATION = 3.0
VAD_TOP_DB   = 20
NR_PROP      = 0.75

# ── Preprocessing functions ───────────────────────────────────

def load_audio(filepath):
    y, sr = librosa.load(filepath, sr=TARGET_SR, mono=True)
    return y, sr

def rms_normalize(y, target_rms=TARGET_RMS):
    current_rms = np.sqrt(np.mean(y ** 2))
    if current_rms < 1e-9:
        return y
    return y * (target_rms / current_rms)

def vad_trim_edges(y, top_db=VAD_TOP_DB):
    # Trims leading/trailing silence ONLY — internal pauses are preserved
    y_trimmed, _ = librosa.effects.trim(y, top_db=top_db)
    return y_trimmed

def reduce_noise(y, sr, prop_decrease=NR_PROP):
    return nr.reduce_noise(y=y, sr=sr, prop_decrease=prop_decrease, stationary=True)

def preprocess_file(filepath, label, dp_id):
    result = {
        "dp_id":           dp_id,
        "original_path":   filepath,
        "preproc_status":  None,
        "preproc_path":    None,
        "duration_before": None,
        "duration_after":  None,
        "dropped_reason":  None,
    }
    try:
        y, sr = load_audio(filepath)
        result["duration_before"] = round(len(y) / sr, 2)

        y = rms_normalize(y)
        y = vad_trim_edges(y)
        y = reduce_noise(y, sr)

        duration_after = len(y) / sr
        result["duration_after"] = round(duration_after, 2)

        if duration_after < MIN_DURATION:
            result["preproc_status"] = "dropped_too_short"
            result["dropped_reason"] = f"Only {duration_after:.1f}s after VAD trim"
            return result

        subdir   = "native" if label == 1 else "non_native"
        out_path = os.path.join(PREPROC_DIR, subdir, f"{dp_id}.wav")
        sf.write(out_path, y, TARGET_SR, subtype="PCM_16")

        result["preproc_status"] = "ok"
        result["preproc_path"]   = out_path

    except Exception as e:
        result["preproc_status"] = "error"
        result["dropped_reason"] = str(e)[:120]

    return result

# ── Load manifest ─────────────────────────────────────────────
print("=" * 58)
print("  STAGE 3 — PREPROCESSING")
print("=" * 58)

df = pd.read_csv(MANIFEST_IN)

processable = df[
    df["status"].isin(["downloaded", "already_exists", "extracted"]) &
    df["save_path"].notna()
].copy()

print(f"\nTotal files in manifest : {len(df)}")
print(f"Files to process        : {len(processable)}")
print(f"Target SR               : {TARGET_SR} Hz")
print(f"Target RMS              : {TARGET_RMS}")
print(f"VAD top_db              : {VAD_TOP_DB}")
print(f"Noise reduction         : {int(NR_PROP*100)}%")
print(f"Min duration gate       : {MIN_DURATION}s\n")

# ── RESUME: skip files whose output WAV already exists on Drive
def get_expected_out_path(row):
    subdir = "native" if row["label"] == 1 else "non_native"
    return os.path.join(PREPROC_DIR, subdir, f"{row['dp_id']}.wav")

processable["expected_out"] = processable.apply(get_expected_out_path, axis=1)
already_done = processable[processable["expected_out"].apply(os.path.exists)]
todo         = processable[~processable["expected_out"].apply(os.path.exists)]

if len(already_done) > 0:
    print(f"⏭️  Skipping {len(already_done)} files already preprocessed on Drive (resume)")
print(f"Processing {len(todo)} files...\n")

# ── Run preprocessing ─────────────────────────────────────────
preproc_results = []

# Add already-done files back as synthetic "ok" results
for _, row in already_done.iterrows():
    preproc_results.append({
        "dp_id":           row["dp_id"],
        "original_path":   row["save_path"],
        "preproc_status":  "ok",
        "preproc_path":    row["expected_out"],
        "duration_before": None,   # not re-measured, not needed
        "duration_after":  None,
        "dropped_reason":  None,
    })

# Process remaining files
for _, row in tqdm(todo.iterrows(), total=len(todo), desc="Preprocessing"):
    res = preprocess_file(
        filepath = row["save_path"],
        label    = row["label"],
        dp_id    = row["dp_id"],
    )
    preproc_results.append(res)

# ── Merge into manifest ───────────────────────────────────────
preproc_df = pd.DataFrame(preproc_results)
df_out     = processable.merge(preproc_df, on="dp_id", how="left")

ok      = df_out[df_out["preproc_status"] == "ok"]
dropped = df_out[df_out["preproc_status"] == "dropped_too_short"]
errors  = df_out[df_out["preproc_status"] == "error"]

# Measure durations for files that were freshly processed (not skipped)
# For skipped files, load duration from existing WAV
def get_duration(path):
    try:
        info = sf.info(path)
        return round(info.duration, 2)
    except Exception:
        return None

ok_with_dur = ok.copy()
mask_no_dur = ok_with_dur["duration_after"].isna()
if mask_no_dur.sum() > 0:
    ok_with_dur.loc[mask_no_dur, "duration_after"] = \
        ok_with_dur.loc[mask_no_dur, "preproc_path"].apply(get_duration)
    df_out.update(ok_with_dur[["dp_id","duration_after"]].set_index("dp_id"))

# Re-read ok after update
ok = df_out[df_out["preproc_status"] == "ok"]

# ── Report ────────────────────────────────────────────────────
print("\n" + "=" * 58)
print("  STAGE 3 — REPORT")
print("=" * 58)
print(f"\n   Successfully preprocessed : {len(ok)}")
print(f"   Dropped (too short)       : {len(dropped)}")
print(f"   Errors                    : {len(errors)}")

if len(dropped) > 0:
    print(f"\n   Dropped files:")
    cols = [c for c in ["dp_id","nativity","dialect","dropped_reason"] if c in dropped.columns]
    print(dropped[cols].to_string(index=False))

if len(errors) > 0:
    print(f"\n   Error files:")
    cols = [c for c in ["dp_id","nativity","dropped_reason"] if c in errors.columns]
    print(errors[cols].to_string(index=False))

ok_dur = ok["duration_after"].dropna()
if len(ok_dur) > 0:
    print(f"\n   Duration after VAD trim:")
    print(f"     mean={ok_dur.mean():.1f}s  min={ok_dur.min():.1f}s  max={ok_dur.max():.1f}s")

print(f"\n   Usable files by class:")
if "nativity" in ok.columns:
    print(ok["nativity"].value_counts().to_string())

est_segs = ok_dur.apply(lambda d: max(0, int((d - 3.0) / 1.0) + 1)).sum()
print(f"\n   Estimated 3s segments (Stage 4 preview): ~{int(est_segs):,}")
print(f"   (3s window, 1s hop, based on trimmed durations)")

# ── Save manifest ─────────────────────────────────────────────
df_out.to_csv(MANIFEST_OUT, index=False)
print(f"\nSaved: {MANIFEST_OUT}")
print("\n✅ Stage 3 complete.")
print("⏭️  Next: Run stage4_splitting.py")


In [ ]:
# ============================================================
# STAGE 4: AUDIO SPLITTING
# Cuts every preprocessed WAV into overlapping 3s segments
# with a 1s hop. Each segment inherits parent file's label.
#
# COLAB SETUP (run this cell first in your notebook):
#   from google.colab import drive
#   drive.mount('/content/drive')
# ============================================================
import os
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm.notebook import tqdm   # notebook-friendly progress bars

# ── Google Drive path config ─────────────────────────────────────
# Change this to match your actual Drive folder name
PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"

MANIFEST_IN  = f"{PROJECT_ROOT}/outputs/stage3_manifest.csv"
MANIFEST_OUT = f"{PROJECT_ROOT}/outputs/stage4_manifest.csv"
SEGMENTS_DIR = f"{PROJECT_ROOT}/data/segments"

WINDOW_SEC = 3.0
HOP_SEC    = 1.0
TARGET_SR  = 16000

os.makedirs(f"{SEGMENTS_DIR}/native",     exist_ok=True)
os.makedirs(f"{SEGMENTS_DIR}/non_native", exist_ok=True)

df     = pd.read_csv(MANIFEST_IN)
usable = df[df["preproc_status"] == "ok"].copy()
print(f"Usable files: {len(usable)}")

# ── SKIP IF ALREADY DONE ─────────────────────────────────────
if os.path.exists(MANIFEST_OUT):
    existing = pd.read_csv(MANIFEST_OUT)
    existing_paths = set(existing["seg_path"].tolist())
    print(f"⏭️  Found existing stage4_manifest.csv ({len(existing)} segments).")
    print(f"   Will skip segments whose WAV already exists on Drive.")
else:
    existing_paths = set()

window_samples = int(WINDOW_SEC * TARGET_SR)   # 48000 samples
hop_samples    = int(HOP_SEC    * TARGET_SR)   # 16000 samples

segment_records = []

for _, row in tqdm(usable.iterrows(), total=len(usable), desc="Splitting"):
    try:
        y, sr = librosa.load(row["preproc_path"], sr=TARGET_SR, mono=True)
    except Exception as e:
        print(f"  Load error {row['dp_id']}: {e}")
        continue

    n_segments = max(0, (len(y) - window_samples) // hop_samples + 1)
    subdir     = "native" if row["label"] == 1 else "non_native"

    for i in range(n_segments):
        start   = i * hop_samples
        end     = start + window_samples
        segment = y[start:end]

        if len(segment) < window_samples:
            segment = np.pad(segment, (0, window_samples - len(segment)))

        seg_id   = f"{row['dp_id']}_seg{i:04d}"
        seg_path = os.path.join(SEGMENTS_DIR, subdir, f"{seg_id}.wav")

        if not os.path.exists(seg_path):
            sf.write(seg_path, segment, TARGET_SR, subtype="PCM_16")

        segment_records.append({
            "seg_id":    seg_id,
            "parent_id": row["dp_id"],
            "seg_index": i,
            "label":     row["label"],
            "nativity":  row["nativity"],
            "dialect":   row["dialect"],
            "source":    row["source"],
            "seg_path":  seg_path,
            "start_sec": round(i * HOP_SEC, 2),
            "end_sec":   round(i * HOP_SEC + WINDOW_SEC, 2),
        })

seg_df = pd.DataFrame(segment_records)
seg_df.to_csv(MANIFEST_OUT, index=False)

print(f"\nTotal segments created : {len(seg_df)}")
print(f"Native segments        : {(seg_df['label']==1).sum()}")
print(f"Non-native segments    : {(seg_df['label']==0).sum()}")
print(f"Saved: {MANIFEST_OUT}")


In [ ]:
# ============================================================
# STAGE 5: DATA AUGMENTATION (training segments only)
# Non-native (minority): 5 augmentations
# Native (majority):     1 augmentation (time stretch only)
# NEVER augment test data.
#
# COLAB NOTE: Augmentation is CPU-heavy. On Colab free tier
# (~17k segments × 5 augs) expect ~45–90 min. Colab Pro/T4
# does not speed this up — librosa runs on CPU only.
# Consider running overnight or in sections using RESUME below.
# ============================================================
import os
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm.notebook import tqdm

# ── Google Drive path config ─────────────────────────────────────
PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"

MANIFEST_IN  = f"{PROJECT_ROOT}/outputs/stage4_manifest.csv"
MANIFEST_OUT = f"{PROJECT_ROOT}/outputs/stage5_manifest.csv"
AUG_DIR      = f"{PROJECT_ROOT}/data/augmented"
TARGET_SR    = 16000
RANDOM_SEED  = 42
np.random.seed(RANDOM_SEED)

os.makedirs(f"{AUG_DIR}/native",     exist_ok=True)
os.makedirs(f"{AUG_DIR}/non_native", exist_ok=True)

seg_df = pd.read_csv(MANIFEST_IN)
print(f"Segments to augment: {len(seg_df)}")

# ── RESUME SUPPORT ───────────────────────────────────────────────
# If the Colab session disconnects mid-run, re-running will skip
# already-completed segments instead of starting over.
already_done = set()
if os.path.exists(MANIFEST_OUT):
    done_df      = pd.read_csv(MANIFEST_OUT)
    already_done = set(done_df[done_df["aug_type"] != "original"]["orig_seg"].dropna())
    print(f"Resuming — {len(already_done)} segments already augmented, skipping.")

# ── Augmentation functions ───────────────────────────────────────
def time_stretch(y, rate):
    return librosa.effects.time_stretch(y, rate=rate)

def pitch_shift(y, sr, steps):
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=steps)

def add_noise(y, sigma=0.005):
    noise = np.random.randn(len(y)) * sigma
    return np.clip(y + noise, -1.0, 1.0)

def save_augmented(y, sr, seg_id, aug_name, label):
    subdir   = "native" if label == 1 else "non_native"
    filename = f"{seg_id}_{aug_name}.wav"
    out_path = os.path.join(AUG_DIR, subdir, filename)
    sf.write(out_path, y, sr, subtype="PCM_16")
    return out_path

# ── Apply augmentations ──────────────────────────────────────────
aug_records = []

for _, row in tqdm(seg_df.iterrows(), total=len(seg_df), desc="Augmenting"):
    # Skip if already processed (resume support)
    if row["seg_id"] in already_done:
        continue

    try:
        y, sr = librosa.load(row["seg_path"], sr=TARGET_SR, mono=True)
    except Exception as e:
        print(f"  Load error {row['seg_id']}: {e}")
        continue

    augmentations = []
    if row["label"] == 0:
        # Non-native (minority class): all 3 augmentation types (5 variants)
        augmentations = [
            ("ts_slow",  time_stretch(y, 0.9)),
            ("ts_fast",  time_stretch(y, 1.1)),
            ("ps_up",    pitch_shift(y, sr, +2)),
            ("ps_down",  pitch_shift(y, sr, -2)),
            ("noise",    add_noise(y)),
        ]
    else:
        # Native (majority class): time stretch only
        augmentations = [
            ("ts_slow",  time_stretch(y, 0.9)),
        ]

    for aug_name, y_aug in augmentations:
        if len(y_aug) > 48000:
            y_aug = y_aug[:48000]
        elif len(y_aug) < 48000:
            y_aug = np.pad(y_aug, (0, 48000 - len(y_aug)))

        out_path = save_augmented(y_aug, sr, row["seg_id"], aug_name, row["label"])
        aug_records.append({
            "seg_id":    f"{row['seg_id']}_{aug_name}",
            "parent_id": row["parent_id"],
            "seg_index": row["seg_index"],
            "label":     row["label"],
            "nativity":  row["nativity"],
            "dialect":   row["dialect"],
            "source":    "augmented",
            "seg_path":  out_path,
            "aug_type":  aug_name,
            "orig_seg":  row["seg_id"],
        })

# ── Combine originals + new augmentations ────────────────────────
aug_df = pd.DataFrame(aug_records)

# If resuming, load existing aug records and append only new ones
if os.path.exists(MANIFEST_OUT) and len(already_done) > 0:
    existing = pd.read_csv(MANIFEST_OUT)
    aug_df   = pd.concat([existing, aug_df], ignore_index=True)
    # Rebuild: originals are from stage4_manifest, augs from aug_df
    originals = seg_df.assign(aug_type="original", orig_seg=None)
    all_segs  = pd.concat(
        [originals, aug_df[aug_df["aug_type"] != "original"]],
        ignore_index=True
    )
else:
    originals = seg_df.assign(aug_type="original", orig_seg=None)
    all_segs  = pd.concat([originals, aug_df], ignore_index=True)

all_segs.to_csv(MANIFEST_OUT, index=False)

print(f"\nOriginal segments  : {len(seg_df)}")
print(f"Augmented segments : {(all_segs['aug_type'] != 'original').sum()}")
print(f"Total segments     : {len(all_segs)}")
print(f"Native total       : {(all_segs['label']==1).sum()}")
print(f"Non-native total   : {(all_segs['label']==0).sum()}")
print(f"Saved: {MANIFEST_OUT}")


In [ ]:
# ============================================================
# STAGE 6A: WHISPER EMBEDDING EXTRACTION
# Model: openai/whisper-large-v3 (encoder only, frozen)
# Output: (N_segments, 1280) numpy array via mean pooling
#
# COLAB NOTES:
#   - Runtime > GPU is STRONGLY recommended (Runtime → Change runtime type)
#   - Model downloads ~3GB on first run, cached in HF_HOME on Drive
#   - BATCH_SIZE=8 works on T4 (16GB VRAM). Use 4 on free tier (12GB)
#   - Checkpoint saves every CHECKPOINT_EVERY batches so a session
#     disconnect doesn't lose all progress
#   - Whisper expects 16kHz mono audio and uses log-Mel spectrogram
#     internally (80 Mel bins, 30s context window)
#   - We use only the ENCODER to get acoustic embeddings (not the
#     decoder, which is for transcription)
# ============================================================
import os
import numpy as np
import pandas as pd
import torch
import librosa
from transformers import WhisperModel, WhisperFeatureExtractor
from tqdm.notebook import tqdm

# ── Google Drive path config ─────────────────────────────────────
PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"

MANIFEST_IN       = f"{PROJECT_ROOT}/outputs/stage5_manifest.csv"
EMBED_DIR         = f"{PROJECT_ROOT}/data/embeddings"
TARGET_SR         = 16000
BATCH_SIZE        = 8         # reduce to 4 if CUDA OOM (free tier Colab)
CHECKPOINT_EVERY  = 50        # save partial results every N batches
DEVICE            = "cuda" if torch.cuda.is_available() else "cpu"

# Whisper model config
WHISPER_MODEL_NAME = "openai/whisper-large-v3"
WHISPER_EMBED_DIM  = 1280     # hidden size of whisper-large-v3 encoder

# ── Persist HuggingFace model cache to Drive (avoids re-downloading each session)
HF_CACHE = f"{PROJECT_ROOT}/hf_cache"
os.environ["HF_HOME"]            = HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE
os.makedirs(HF_CACHE,  exist_ok=True)
os.makedirs(EMBED_DIR, exist_ok=True)

print(f"Using device: {DEVICE}")
if DEVICE == "cpu":
    print("WARNING: No GPU detected. Embedding extraction will be very slow on CPU.")
    print("Go to Runtime → Change runtime type → GPU (T4)")

# ── SKIP IF ALREADY DONE ─────────────────────────────────────
_final_embeds = f"{EMBED_DIR}/whisper_embeddings.npy"
_final_ids    = f"{EMBED_DIR}/whisper_seg_ids.npy"
if os.path.exists(_final_embeds) and os.path.exists(_final_ids):
    _X = np.load(_final_embeds)
    print(f"⏭️  Embeddings already exist: {_X.shape} — skipping extraction.")
    print(f"   Delete {_final_embeds} to force re-run.")
    print("\n✅ Stage 6A already done — skipped.")
    raise SystemExit(0)

# ── Load Whisper model (encoder only) ────────────────────────────
print(f"Loading Whisper model: {WHISPER_MODEL_NAME} (downloads ~3GB if not cached)...")
feature_extractor = WhisperFeatureExtractor.from_pretrained(WHISPER_MODEL_NAME)
model = WhisperModel.from_pretrained(WHISPER_MODEL_NAME)
model = model.to(DEVICE)
model.eval()

# We only need the encoder — freeze all parameters
for param in model.parameters():
    param.requires_grad = False

print("Whisper model loaded (encoder will be used for embeddings).")

seg_df = pd.read_csv(MANIFEST_IN)
print(f"Total segments to embed: {len(seg_df)}")

# ── CHECKPOINT / RESUME ──────────────────────────────────────────
checkpoint_embeds_path  = f"{EMBED_DIR}/whisper_embeddings_partial.npy"
checkpoint_segids_path  = f"{EMBED_DIR}/whisper_seg_ids_partial.npy"
start_batch = 0

all_embeddings = []
all_seg_ids    = []

if os.path.exists(checkpoint_embeds_path) and os.path.exists(checkpoint_segids_path):
    prev_embeds  = np.load(checkpoint_embeds_path)
    prev_seg_ids = np.load(checkpoint_segids_path, allow_pickle=True)
    all_embeddings.append(prev_embeds)
    all_seg_ids.extend(prev_seg_ids.tolist())
    start_batch  = (len(prev_seg_ids) // BATCH_SIZE)
    print(f"Resuming from batch {start_batch} ({len(prev_seg_ids)} segments already done)")

# ── Embedding extraction ─────────────────────────────────────────
def extract_embedding_batch(paths):
    """
    Load audio files, compute log-Mel spectrograms via Whisper's
    feature extractor, pass through the Whisper encoder, and
    mean-pool the encoder hidden states to get a fixed-size
    embedding per audio segment.
    """
    waveforms = []
    for path in paths:
        y, _ = librosa.load(path, sr=TARGET_SR, mono=True)
        # Whisper expects up to 30s of audio at 16kHz
        # Our segments are 3s = 48000 samples — well within limit
        if len(y) > TARGET_SR * 30:
            y = y[:TARGET_SR * 30]
        waveforms.append(y)

    # Whisper feature extractor produces log-Mel spectrograms
    # padding/truncation to 30s is handled automatically
    inputs = feature_extractor(
        waveforms,
        sampling_rate=TARGET_SR,
        return_tensors="pt",
        padding=True,
    )
    input_features = inputs.input_features.to(DEVICE)  # (batch, 80, 3000)

    with torch.no_grad():
        # Use only the encoder to get hidden states
        encoder_outputs = model.encoder(input_features)
        hidden_states = encoder_outputs.last_hidden_state  # (batch, seq_len, 1280)
        # Mean pool over the time dimension → (batch, 1280)
        embeddings = hidden_states.mean(dim=1)

    return embeddings.cpu().numpy()

paths   = seg_df["seg_path"].tolist()
seg_ids = seg_df["seg_id"].tolist()
errors  = []

# Skip already-processed batches
paths_todo   = paths[start_batch * BATCH_SIZE:]
seg_ids_todo = seg_ids[start_batch * BATCH_SIZE:]

for i in tqdm(range(0, len(paths_todo), BATCH_SIZE), desc="Extracting Whisper embeddings"):
    batch_paths   = paths_todo[i : i + BATCH_SIZE]
    batch_seg_ids = seg_ids_todo[i : i + BATCH_SIZE]

    try:
        embeds = extract_embedding_batch(batch_paths)
        all_embeddings.append(embeds)
        all_seg_ids.extend(batch_seg_ids)
    except RuntimeError as e:
        if "CUDA out of memory" in str(e):
            print(f"\nOOM at batch {i} — try reducing BATCH_SIZE to {BATCH_SIZE // 2}")
            torch.cuda.empty_cache()
            raise
        print(f"  Batch {i} error: {e}")
        errors.append({"batch_start": i, "error": str(e)})
        all_embeddings.append(np.zeros((len(batch_paths), WHISPER_EMBED_DIM)))
        all_seg_ids.extend(batch_seg_ids)

    # Save checkpoint periodically (protects against Colab session drops)
    if (i // BATCH_SIZE + 1) % CHECKPOINT_EVERY == 0:
        partial_embeds = np.vstack(all_embeddings)
        np.save(checkpoint_embeds_path, partial_embeds)
        np.save(checkpoint_segids_path, np.array(all_seg_ids))
        print(f"  Checkpoint saved at batch {i // BATCH_SIZE + 1} ({len(all_seg_ids)} segments)")

# ── Final save ───────────────────────────────────────────────────
X_whisper = np.vstack(all_embeddings)   # (N_segments, 1280)
np.save(f"{EMBED_DIR}/whisper_embeddings.npy", X_whisper)
np.save(f"{EMBED_DIR}/whisper_seg_ids.npy",    np.array(all_seg_ids))

# Clean up checkpoint files
for f in [checkpoint_embeds_path, checkpoint_segids_path]:
    if os.path.exists(f):
        os.remove(f)

print(f"\nEmbeddings shape : {X_whisper.shape}")
print(f"Saved: {EMBED_DIR}/whisper_embeddings.npy")
if errors:
    print(f"WARNING: Errors in {len(errors)} batches")

In [ ]:
# ============================================================
# STAGE 6B: PROSODIC FEATURE EXTRACTION
# Runs on UTTERANCE level (stage3_manifest).
# All segments from the same recording share the same 6-dim
# prosodic vector (looked up by parent_id in Stage 7).
#
# COLAB NOTES:
#   - parselmouth must be installed: pip install praat-parselmouth
#   - This runs on CPU — no GPU benefit here
#   - Can be run in parallel with Stage 6A (different inputs/outputs)
# ============================================================
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# Install parselmouth if not already installed (safe to re-run)
try:
    import parselmouth
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "praat-parselmouth", "-q"], check=True)
    import parselmouth

# ── Google Drive path config ─────────────────────────────────────
PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"

MANIFEST_IN  = f"{PROJECT_ROOT}/outputs/stage3_manifest.csv"
PROSODIC_OUT = f"{PROJECT_ROOT}/data/embeddings/prosodic_features.csv"

os.makedirs(f"{PROJECT_ROOT}/data/embeddings", exist_ok=True)

# ── SKIP IF ALREADY DONE ─────────────────────────────────────
if os.path.exists(PROSODIC_OUT):
    existing = pd.read_csv(PROSODIC_OUT)
    print(f"⏭️  prosodic_features.csv already exists ({len(existing)} utterances) — skipping.")
    print(f"   Delete {PROSODIC_OUT} to force re-run.")
    print("\n✅ Stage 6B already done — skipped.")
    raise SystemExit(0)


def extract_prosodic_features(audio_path):
    """
    Extract 6 prosodic features from a full utterance WAV file.
    Returns np.array of shape (6,) or zeros on failure.
    Features: [f0_mean, f0_var, speaking_rate, pause_freq,
               pause_dur_mean, npvi]
    """
    try:
        sound    = parselmouth.Sound(audio_path)
        duration = sound.end_time - sound.start_time

        # ── F0 mean and variance ──────────────────────────────
        pitch     = sound.to_pitch(time_step=0.01, pitch_floor=75, pitch_ceiling=600)
        f0_vals   = pitch.selected_array['frequency']
        f0_voiced = f0_vals[f0_vals > 0]

        f0_mean = float(np.mean(f0_voiced)) if len(f0_voiced) > 0 else 0.0
        f0_var  = float(np.var(f0_voiced))  if len(f0_voiced) > 0 else 0.0

        # ── Speaking rate (syllable-proxy peaks/sec) ──────────
        intensity  = sound.to_intensity(minimum_pitch=75)
        int_values = intensity.values.flatten()
        int_mean   = float(np.mean(int_values))

        peaks = 0
        for j in range(1, len(int_values) - 1):
            if (int_values[j] > int_mean and
                    int_values[j] > int_values[j-1] and
                    int_values[j] > int_values[j+1]):
                peaks += 1

        speaking_rate = peaks / duration if duration > 0 else 0.0

        # ── Pause frequency and mean pause duration ───────────
        pause_threshold_sec = 0.2

        f0_times  = [pitch.get_time_from_frame_number(i+1)
                     for i in range(pitch.get_number_of_frames())]
        is_voiced = [pitch.get_value_at_time(t) is not None and
                     pitch.get_value_at_time(t) > 0
                     for t in f0_times]

        pauses      = []
        in_pause    = False
        pause_start = 0.0

        for t, voiced in zip(f0_times, is_voiced):
            if not voiced and not in_pause:
                in_pause    = True
                pause_start = t
            elif voiced and in_pause:
                pause_dur = t - pause_start
                if pause_dur >= pause_threshold_sec:
                    pauses.append(pause_dur)
                in_pause = False

        pause_freq     = len(pauses) / duration if duration > 0 else 0.0
        pause_dur_mean = float(np.mean(pauses)) if pauses else 0.0

        # ── nPVI ──────────────────────────────────────────────
        vowel_durations = []
        in_vowel        = False
        vowel_start     = 0.0

        for t, voiced in zip(f0_times, is_voiced):
            if voiced and not in_vowel:
                in_vowel    = True
                vowel_start = t
            elif not voiced and in_vowel:
                vowel_dur = t - vowel_start
                if vowel_dur >= 0.03:
                    vowel_durations.append(vowel_dur)
                in_vowel = False

        if len(vowel_durations) >= 2:
            durs  = np.array(vowel_durations)
            dk    = durs[:-1]
            dk1   = durs[1:]
            denom = (dk + dk1) / 2
            valid = denom > 0
            npvi  = float(np.mean(np.abs(dk[valid] - dk1[valid]) / denom[valid]) * 100)
        else:
            npvi = 0.0

        return np.array([f0_mean, f0_var, speaking_rate,
                         pause_freq, pause_dur_mean, npvi], dtype=np.float32)

    except Exception as e:
        print(f"  Prosodic extraction failed for {audio_path}: {e}")
        return np.zeros(6, dtype=np.float32)


# ── Run extraction ───────────────────────────────────────────────
df     = pd.read_csv(MANIFEST_IN)
usable = df[df["preproc_status"] == "ok"].copy()
print(f"Extracting prosodic features from {len(usable)} utterances...")

records = []
for _, row in tqdm(usable.iterrows(), total=len(usable), desc="Prosodic features"):
    features = extract_prosodic_features(row["preproc_path"])
    records.append({
        "dp_id":          row["dp_id"],
        "label":          row["label"],
        "f0_mean":        features[0],
        "f0_var":         features[1],
        "speaking_rate":  features[2],
        "pause_freq":     features[3],
        "pause_dur_mean": features[4],
        "npvi":           features[5],
    })

prosodic_df = pd.DataFrame(records)
prosodic_df.to_csv(PROSODIC_OUT, index=False)

print(f"\nProsodic features shape : {prosodic_df.shape}")
print(f"Saved: {PROSODIC_OUT}")

print("\nFeature means by class:")
print(prosodic_df.groupby("label")[
    ["f0_mean","f0_var","speaking_rate","pause_freq","pause_dur_mean","npvi"]
].mean().round(3).to_string())


In [ ]:
# ============================================================
# STAGE 7: FEATURE FUSION
# Concatenates Whisper (1280-dim) + prosodic (6-dim) per segment
# → 1286-dim feature vector.
# StandardScaler fitted on TRAIN only, applied to test.
# Group-level split by parent_id — no data leakage.
#
# COLAB NOTES:
#   - Pure numpy/sklearn — no GPU needed, runs fast on CPU
#   - Loads .npy files from Drive (may be slow if files are large;
#     consider copying to /content/ first for faster I/O)
# ============================================================
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
import joblib

# ── Google Drive path config ─────────────────────────────────────
PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"

EMBED_DIR    = f"{PROJECT_ROOT}/data/embeddings"
MANIFEST_IN  = f"{PROJECT_ROOT}/outputs/stage5_manifest.csv"
FEATURES_OUT = f"{PROJECT_ROOT}/outputs/stage7_features.npz"

os.makedirs(f"{PROJECT_ROOT}/outputs", exist_ok=True)

# ── Load Whisper embeddings ──────────────────────────────────────
print("Loading Whisper embeddings...")
X_whisper   = np.load(f"{EMBED_DIR}/whisper_embeddings.npy")
seg_ids_wh  = np.load(f"{EMBED_DIR}/whisper_seg_ids.npy", allow_pickle=True)
print(f"  Embeddings shape: {X_whisper.shape}")

WHISPER_EMBED_DIM = X_whisper.shape[1]  # 1280 for whisper-large-v3

# ── Load prosodic features ────────────────────────────────────────
print("Loading prosodic features...")
prosodic_df  = pd.read_csv(f"{EMBED_DIR}/prosodic_features.csv")
prosodic_map = prosodic_df.set_index("dp_id")[
    ["f0_mean","f0_var","speaking_rate","pause_freq","pause_dur_mean","npvi"]
].to_dict("index")
print(f"  Prosodic utterances: {len(prosodic_map)}")

# ── Load segment manifest ─────────────────────────────────────────
seg_df = pd.read_csv(MANIFEST_IN)
print(f"  Total segments in manifest: {len(seg_df)}")

seg_id_to_idx = {sid: i for i, sid in enumerate(seg_ids_wh)}

fused_features  = []
labels          = []
parent_ids      = []
seg_ids_final   = []
aug_types_list  = []
seg_indices_list = []
missing_whisper = 0
missing_pros    = 0

PROSODIC_DIM = 6
FUSED_DIM    = WHISPER_EMBED_DIM + PROSODIC_DIM  # 1286

for _, row in seg_df.iterrows():
    seg_id = row["seg_id"]

    if seg_id not in seg_id_to_idx:
        missing_whisper += 1
        continue

    wh_embed    = X_whisper[seg_id_to_idx[seg_id]]   # (1280,)
    parent_id   = row["parent_id"]
    base_parent = parent_id.split("_aug")[0] if "_aug" in parent_id else parent_id

    if base_parent in prosodic_map:
        pros_vec = np.array(list(prosodic_map[base_parent].values()), dtype=np.float32)
    else:
        pros_vec = np.zeros(PROSODIC_DIM, dtype=np.float32)
        missing_pros += 1

    fused_features.append(np.concatenate([wh_embed, pros_vec]))  # (1286,)
    labels.append(row["label"])
    parent_ids.append(parent_id)
    seg_ids_final.append(seg_id)
    aug_types_list.append(row.get("aug_type", "original"))
    seg_indices_list.append(int(row.get("seg_index", 0)))

X      = np.array(fused_features)
y      = np.array(labels)
groups = np.array(parent_ids)
aug_types_arr   = np.array(aug_types_list)
seg_indices_arr = np.array(seg_indices_list, dtype=np.int32)

print(f"\nFused feature matrix shape : {X.shape}")
print(f"  Whisper embedding dim    : {WHISPER_EMBED_DIM}")
print(f"  Prosodic feature dim     : {PROSODIC_DIM}")
print(f"  Total fused dim          : {FUSED_DIM}")
print(f"Label distribution — Native: {(y==1).sum()}, Non-Native: {(y==0).sum()}")
if missing_whisper:
    print(f"WARNING: {missing_whisper} segments skipped (missing Whisper embedding)")
if missing_pros:
    print(f"WARNING: {missing_pros} segments used zero prosodic vector")

# ── Group-level Train/Test Split ──────────────────────────────────
# Split by parent_id so no recording appears in both train and test
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"\nTrain segments : {X_train.shape[0]}")
print(f"Test segments  : {X_test.shape[0]}")
print(f"Train — Native: {(y_train==1).sum()}, Non-Native: {(y_train==0).sum()}")
print(f"Test  — Native: {(y_test==1).sum()},  Non-Native: {(y_test==0).sum()}")

# ── Normalize — fit on train only ────────────────────────────────
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)    # transform only — no refit

# ── Save ──────────────────────────────────────────────────────────
np.savez(
    FEATURES_OUT,
    X_train=X_train_sc,
    X_test=X_test_sc,
    y_train=y_train,
    y_test=y_test,
    train_idx=train_idx,
    test_idx=test_idx,
    train_parent_ids=groups[train_idx],
    test_parent_ids=groups[test_idx],
    train_aug_types=aug_types_arr[train_idx],
    test_aug_types=aug_types_arr[test_idx],
    train_seg_indices=seg_indices_arr[train_idx],
    test_seg_indices=seg_indices_arr[test_idx],
)
joblib.dump(scaler, f"{PROJECT_ROOT}/outputs/scaler.joblib")

print(f"\nFeatures saved : {FEATURES_OUT}")
print(f"Scaler saved   : {PROJECT_ROOT}/outputs/scaler.joblib")

In [ ]:
# ============================================================
# STAGE 8: CLASSIFICATION
# Primary: LSTM feature extractor + Random Forest classifier
# Baseline: Logistic Regression
#
# Architecture:
#   1. Group segment-level features into recording-level
#      sequences (keyed by parent_id + aug_type, sorted
#      by seg_index so temporal order is preserved).
#   2. Bidirectional LSTM encodes each variable-length
#      sequence → fixed 256-dim hidden-state vector.
#   3. Random Forest classifies the LSTM-extracted features.
#
# NOTE: Input features are Whisper (1280-dim) + prosodic (6-dim)
#       = 1286-dim per segment (from Stage 7).
#
# COLAB NOTES:
#   - LSTM training benefits from GPU (Runtime → GPU)
#   - Random Forest runs on CPU (sklearn)
#   - Expect ~5–20 min total depending on dataset size
#   - Models saved to Drive immediately after training
# ============================================================
import os
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

# ── Google Drive path config ─────────────────────────────────────
PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"

FEATURES_IN = f"{PROJECT_ROOT}/outputs/stage7_features.npz"
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(f"{PROJECT_ROOT}/outputs", exist_ok=True)

# ── Load features ─────────────────────────────────────────────────
print("Loading features from Drive...")
data    = np.load(FEATURES_IN, allow_pickle=True)
X_train = data["X_train"]
X_test  = data["X_test"]
y_train = data["y_train"]
y_test  = data["y_test"]

train_parent_ids  = data["train_parent_ids"]
test_parent_ids   = data["test_parent_ids"]
train_aug_types   = data["train_aug_types"]
test_aug_types    = data["test_aug_types"]
train_seg_indices = data["train_seg_indices"]
test_seg_indices  = data["test_seg_indices"]

INPUT_DIM = X_train.shape[1]   # 1286 (Whisper 1280 + prosodic 6)

print(f"Training on {X_train.shape[0]} segments, {INPUT_DIM} features")
print(f"Test set: {X_test.shape[0]} segments")
print(f"Train label distribution — Native: {(y_train==1).sum()}, Non-Native: {(y_train==0).sum()}")
print(f"Using device: {DEVICE}")

# ── SKIP IF ALREADY TRAINED ───────────────────────────────────────
_lstm_path = f"{PROJECT_ROOT}/outputs/lstm_encoder.pt"
_rf_path   = f"{PROJECT_ROOT}/outputs/rf_model.joblib"
if os.path.exists(_lstm_path) and os.path.exists(_rf_path):
    print(f"\n⏭️  Models already exist — skipping training.")
    lstm_config = joblib.load(f"{PROJECT_ROOT}/outputs/lstm_config.joblib")
    test_data   = np.load(f"{PROJECT_ROOT}/outputs/stage8_test_data.npz", allow_pickle=True)
    rf = joblib.load(_rf_path)
    y_pred = rf.predict(test_data["X_test_lstm"])
    y_true = test_data["y_test_seq"]
    print(f"Test F1  : {f1_score(y_true, y_pred):.4f}")
    print("\nClassification report (test set — recording level):")
    print(classification_report(y_true, y_pred, target_names=["Non-Native", "Native"]))
    print("   Delete outputs/lstm_encoder.pt and outputs/rf_model.joblib to force re-training.")
    print("\n✅ Stage 8 already done — skipped.")
    raise SystemExit(0)

# ══════════════════════════════════════════════════════════════════
# GROUP SEGMENTS INTO RECORDING-LEVEL SEQUENCES
# ══════════════════════════════════════════════════════════════════
def group_into_sequences(X, y, parent_ids, aug_types, seg_indices):
    """
    Group segment features into recording-level sequences.
    Key = (parent_id, aug_type) so each augmentation variant
    forms its own temporal sequence sorted by seg_index.
    """
    seq_keys    = np.array([f"{pid}__{aug}" for pid, aug
                            in zip(parent_ids, aug_types)])
    unique_keys = np.unique(seq_keys)

    sequences    = []
    seq_labels   = []
    seq_lengths  = []
    seq_key_list = []

    for key in unique_keys:
        mask       = seq_keys == key
        indices    = seg_indices[mask]
        sort_order = np.argsort(indices)

        seq   = X[mask][sort_order]        # (n_segs, 1286)
        label = int(y[mask][0])

        sequences.append(torch.FloatTensor(seq))
        seq_labels.append(label)
        seq_lengths.append(len(seq))
        seq_key_list.append(key)

    return sequences, np.array(seq_labels), seq_lengths, seq_key_list


train_seqs, y_train_seq, train_lens, train_keys = group_into_sequences(
    X_train, y_train, train_parent_ids, train_aug_types, train_seg_indices
)
test_seqs, y_test_seq, test_lens, test_keys = group_into_sequences(
    X_test, y_test, test_parent_ids, test_aug_types, test_seg_indices
)

print(f"\nRecording-level sequences:")
print(f"  Train: {len(train_seqs)} sequences "
      f"(from {len(np.unique(train_parent_ids))} recordings)")
print(f"  Test : {len(test_seqs)} sequences "
      f"(from {len(np.unique(test_parent_ids))} recordings)")
print(f"  Train — Native: {(y_train_seq==1).sum()}, "
      f"Non-Native: {(y_train_seq==0).sum()}")
print(f"  Test  — Native: {(y_test_seq==1).sum()},  "
      f"Non-Native: {(y_test_seq==0).sum()}")
print(f"  Avg sequence length — Train: {np.mean(train_lens):.1f}, "
      f"Test: {np.mean(test_lens):.1f}")

# ══════════════════════════════════════════════════════════════════
# LSTM ENCODER
# ══════════════════════════════════════════════════════════════════
HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT    = 0.3
LSTM_FEAT_DIM = HIDDEN_DIM * 2   # bidirectional → 256


class SequenceDataset(Dataset):
    def __init__(self, sequences, labels, lengths):
        self.sequences = sequences
        self.labels    = torch.FloatTensor(labels)
        self.lengths   = lengths

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx], self.lengths[idx]


def collate_fn(batch):
    seqs, labels, lengths = zip(*batch)
    padded = pad_sequence(seqs, batch_first=True)      # (B, max_len, 1286)
    return padded, torch.stack(labels), torch.LongTensor(lengths)


class LSTMEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim, hidden_dim, num_layers,
            batch_first=True, dropout=dropout, bidirectional=True,
        )
        # Classification head — used during LSTM pre-training only
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def encode(self, x, lengths):
        """Return the fixed-size hidden-state vector (hidden_dim*2)."""
        packed = pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (h_n, _) = self.lstm(packed)
        # Concat final forward + backward hidden states
        h_final = torch.cat([h_n[-2], h_n[-1]], dim=1)   # (B, hidden*2)
        return h_final

    def forward(self, x, lengths):
        h = self.encode(x, lengths)
        logits = self.classifier(h).squeeze(-1)
        return logits, h


# ── Build model ───────────────────────────────────────────────────
model     = LSTMEncoder(INPUT_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

# Class-balanced loss
n_pos = int((y_train_seq == 1).sum())
n_neg = int((y_train_seq == 0).sum())
pos_weight = torch.tensor([n_neg / max(n_pos, 1)]).to(DEVICE)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

train_dataset = SequenceDataset(train_seqs, y_train_seq, train_lens)
train_loader  = DataLoader(
    train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn
)

# ── Train LSTM ────────────────────────────────────────────────────
EPOCHS   = 50
PATIENCE = 10

best_loss        = float("inf")
patience_counter = 0

print(f"\nTraining LSTM encoder ({EPOCHS} epochs max, patience={PATIENCE})...")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    n_batches  = 0

    for batch_x, batch_y, batch_lens in train_loader:
        batch_x    = batch_x.to(DEVICE)
        batch_y    = batch_y.to(DEVICE)
        batch_lens = batch_lens.to(DEVICE)

        logits, _ = model(batch_x, batch_lens)
        loss = criterion(logits, batch_y)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches  += 1

    avg_loss = total_loss / n_batches

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:>3}/{EPOCHS}: loss={avg_loss:.4f}")

    if avg_loss < best_loss - 1e-4:
        best_loss = avg_loss
        patience_counter = 0
        torch.save(model.state_dict(), _lstm_path)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"  Early stopping at epoch {epoch+1}")
            break

# Reload best weights
model.load_state_dict(torch.load(_lstm_path, map_location=DEVICE))
print(f"Best LSTM training loss: {best_loss:.4f}")

# ── Extract LSTM features ─────────────────────────────────────────
def extract_lstm_features(mdl, sequences, lengths, device):
    mdl.eval()
    features = []
    with torch.no_grad():
        for seq, length in zip(sequences, lengths):
            x = seq.unsqueeze(0).to(device)          # (1, seq_len, 1286)
            l = torch.LongTensor([length]).to(device)
            h = mdl.encode(x, l)                     # (1, 256)
            features.append(h.cpu().numpy().squeeze(0))
    return np.array(features)

print("\nExtracting LSTM features...")
X_train_lstm = extract_lstm_features(model, train_seqs, train_lens, DEVICE)
X_test_lstm  = extract_lstm_features(model, test_seqs,  test_lens,  DEVICE)
print(f"  Train LSTM features: {X_train_lstm.shape}")
print(f"  Test  LSTM features: {X_test_lstm.shape}")

# ══════════════════════════════════════════════════════════════════
# RANDOM FOREST CLASSIFIER
# ══════════════════════════════════════════════════════════════════
print("\nTraining Random Forest on LSTM-extracted features...")
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train_lstm, y_train_seq)

# ── Evaluate ──────────────────────────────────────────────────────
y_pred_train = rf.predict(X_train_lstm)
y_pred_test  = rf.predict(X_test_lstm)

train_f1 = f1_score(y_train_seq, y_pred_train)
test_f1  = f1_score(y_test_seq,  y_pred_test)

print(f"\nTrain F1 : {train_f1:.4f}")
print(f"Test F1  : {test_f1:.4f}")

print("\nClassification report (test set — recording level):")
print(classification_report(y_test_seq, y_pred_test,
                            target_names=["Non-Native", "Native"]))

# ── Baseline: Logistic Regression ─────────────────────────────────
print("Training Logistic Regression baseline on LSTM features...")
lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
lr.fit(X_train_lstm, y_train_seq)

lr_f1 = f1_score(y_test_seq, lr.predict(X_test_lstm))
print(f"Logistic Regression Test F1 : {lr_f1:.4f}")
print(f"RF improvement over LR      : +{(test_f1 - lr_f1):.4f}")

# ── Save everything ───────────────────────────────────────────────
# LSTM config (needed to reconstruct model in Stage 9/10)
lstm_config = {
    "input_dim":  INPUT_DIM,
    "hidden_dim": HIDDEN_DIM,
    "num_layers": NUM_LAYERS,
    "dropout":    DROPOUT,
}
joblib.dump(lstm_config, f"{PROJECT_ROOT}/outputs/lstm_config.joblib")
joblib.dump(rf,          f"{PROJECT_ROOT}/outputs/rf_model.joblib")
joblib.dump(lr,          f"{PROJECT_ROOT}/outputs/lr_baseline.joblib")

# Recording-level test data (for Stage 9 evaluation)
np.savez(
    f"{PROJECT_ROOT}/outputs/stage8_test_data.npz",
    X_test_lstm=X_test_lstm,
    y_test_seq=y_test_seq,
    test_keys=np.array(test_keys),
)

print("\nModels saved to Drive:")
print(f"  {PROJECT_ROOT}/outputs/lstm_encoder.pt")
print(f"  {PROJECT_ROOT}/outputs/lstm_config.joblib")
print(f"  {PROJECT_ROOT}/outputs/rf_model.joblib")
print(f"  {PROJECT_ROOT}/outputs/lr_baseline.joblib")
print(f"  {PROJECT_ROOT}/outputs/stage8_test_data.npz")

In [ ]:
# ============================================================
# STAGE 9: EVALUATION
# Metrics: Accuracy, F1 (primary), Precision, Recall,
#          ROC AUC, EER, Confusion Matrix, 95% CI
#
# Evaluation is at RECORDING level — one prediction per
# recording sequence produced by LSTM encoder → RF.
#
# COLAB NOTES:
#   - Pure CPU — no GPU needed, runs in seconds
#   - Results saved to Drive as stage9_metrics.csv
# ============================================================
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import (f1_score, confusion_matrix, accuracy_score,
                              roc_curve, precision_score, recall_score,
                              roc_auc_score)
from scipy import stats

# ── Google Drive path config ─────────────────────────────────────
PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"

os.makedirs(f"{PROJECT_ROOT}/outputs", exist_ok=True)

# ── Load recording-level test data and RF model ───────────────────
print("Loading model and test data...")
test_data   = np.load(f"{PROJECT_ROOT}/outputs/stage8_test_data.npz",
                       allow_pickle=True)
X_test_lstm = test_data["X_test_lstm"]
y_test      = test_data["y_test_seq"]
test_keys   = test_data["test_keys"]

rf = joblib.load(f"{PROJECT_ROOT}/outputs/rf_model.joblib")

y_pred   = rf.predict(X_test_lstm)
y_proba  = rf.predict_proba(X_test_lstm)
y_scores = y_proba[:, 1]   # P(Native)

# ── Metrics ───────────────────────────────────────────────────────
acc       = accuracy_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall    = recall_score(y_test, y_pred, zero_division=0)
auc       = roc_auc_score(y_test, y_scores)
cm        = confusion_matrix(y_test, y_pred)

# EER: threshold where FPR == FRR
fpr, tpr, thresholds = roc_curve(y_test, y_scores)
frr     = 1 - tpr
eer_idx = np.argmin(np.abs(fpr - frr))
eer     = (fpr[eer_idx] + frr[eer_idx]) / 2
eer_thr = thresholds[eer_idx]

# 95% Binomial CI on accuracy
n = len(y_test)
ci_low, ci_high = stats.binom.interval(0.95, n, acc)
ci_low  /= n
ci_high /= n

# ── Print report ──────────────────────────────────────────────────
sep = "=" * 55
print(f"\n{sep}")
print("  EVALUATION REPORT — Stage 9  (LSTM + Random Forest)")
print(sep)
print(f"\n  Test set size : {n} recording sequences")
print(f"\n  Accuracy  : {acc*100:.2f}%")
print(f"  F1 Score  : {f1:.4f}  ← primary metric")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  ROC AUC   : {auc:.4f}")
print(f"  EER       : {eer*100:.2f}%  (threshold={eer_thr:.4f})")
print(f"\n  95% CI on Accuracy: [{ci_low*100:.1f}%, {ci_high*100:.1f}%]")
print(f"\n  Confusion Matrix:")
print(f"  (Rows = True class, Cols = Predicted class)")
print(f"              Pred Non-Native  Pred Native")
print(f"  True Non-Nat   {cm[0,0]:>6}          {cm[0,1]:>6}")
print(f"  True Native    {cm[1,0]:>6}          {cm[1,1]:>6}")
print(f"\n  False Positives (non-native → native) : {cm[0,1]}")
print(f"  False Negatives (native → non-native) : {cm[1,0]}")
print(sep)

if eer < 0.05:
    print("  EER < 5%   → Excellent, production-grade")
elif eer < 0.10:
    print("  EER 5–10%  → Good, solid research-grade")
elif eer < 0.20:
    print("  EER 10–20% → Acceptable")
else:
    print("  EER > 20%  → Needs improvement")
print(sep)

# ── Dialect breakdown ─────────────────────────────────────────────
try:
    seg_df = pd.read_csv(f"{PROJECT_ROOT}/outputs/stage5_manifest.csv")
    parent_to_dialect = (seg_df.drop_duplicates("parent_id")
                         .set_index("parent_id")["dialect"].to_dict())

    test_parents  = [k.split("__")[0] for k in test_keys]
    test_dialects = [parent_to_dialect.get(p, "unknown") for p in test_parents]

    test_df = pd.DataFrame({
        "dialect": test_dialects,
        "y_true":  y_test,
        "y_pred":  y_pred,
        "correct": (y_pred == y_test),
    })

    print("\n  Accuracy by dialect (recording level):")
    for dialect, group in test_df.groupby("dialect"):
        acc_d = group["correct"].mean()
        n_d   = len(group)
        print(f"    {dialect:<20} {acc_d*100:.1f}%  (n={n_d})")
except Exception as e:
    print(f"\n  (Could not compute dialect breakdown: {e})")

# ── Save metrics ──────────────────────────────────────────────────
metrics = {
    "accuracy":      acc,
    "f1_score":      f1,
    "precision":     precision,
    "recall":        recall,
    "roc_auc":       auc,
    "eer":           eer,
    "eer_threshold": eer_thr,
    "ci_low":        ci_low,
    "ci_high":       ci_high,
    "n_test":        n,
    "tn": cm[0,0], "fp": cm[0,1],
    "fn": cm[1,0], "tp": cm[1,1],
}
pd.DataFrame([metrics]).to_csv(f"{PROJECT_ROOT}/outputs/stage9_metrics.csv", index=False)
print(f"\nSaved: {PROJECT_ROOT}/outputs/stage9_metrics.csv")


In [ ]:
# ============================================================
# STAGE 10: OUTPUT GENERATION
# Deliverables:
#   1. outputs/predictions.csv
#   2. outputs/technical_report.md
#
# COLAB NOTES:
#   - Runs in seconds — pure CPU, no heavy computation
#   - Both output files are saved directly to Drive
# ============================================================
import os
import numpy as np
import pandas as pd
import joblib
from datetime import date

# ── Google Drive path config ─────────────────────────────────────
PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"

os.makedirs(f"{PROJECT_ROOT}/outputs", exist_ok=True)

# ── Load everything ───────────────────────────────────────────────
print("Loading models, features, and metrics...")
test_data    = np.load(f"{PROJECT_ROOT}/outputs/stage8_test_data.npz",
                        allow_pickle=True)
X_test_lstm  = test_data["X_test_lstm"]
y_test       = test_data["y_test_seq"]
test_keys    = test_data["test_keys"]

rf      = joblib.load(f"{PROJECT_ROOT}/outputs/rf_model.joblib")
metrics = pd.read_csv(f"{PROJECT_ROOT}/outputs/stage9_metrics.csv").iloc[0]

# ── Predictions ───────────────────────────────────────────────────
y_pred          = rf.predict(X_test_lstm)
probs           = rf.predict_proba(X_test_lstm)
confidence      = probs.max(axis=1)
prob_native     = probs[:, 1]
prob_non_native = probs[:, 0]

# ── Map test_keys back to metadata ────────────────────────────────
try:
    seg_df = pd.read_csv(f"{PROJECT_ROOT}/outputs/stage5_manifest.csv")
    parent_to_dialect = (seg_df.drop_duplicates("parent_id")
                         .set_index("parent_id")["dialect"].to_dict())

    recording_ids = [k.split("__")[0] for k in test_keys]
    aug_types     = [k.split("__")[1] if "__" in k else "original"
                     for k in test_keys]
    dialects      = [parent_to_dialect.get(r, "unknown")
                     for r in recording_ids]
except Exception as e:
    print(f"Warning: could not load segment info ({e}). Using generic IDs.")
    recording_ids = [f"rec_{i:03d}" for i in range(len(y_test))]
    aug_types     = ["unknown"] * len(y_test)
    dialects      = ["unknown"] * len(y_test)

# ── predictions.csv ───────────────────────────────────────────────
results = pd.DataFrame({
    "recording_id":     recording_ids,
    "aug_type":         aug_types,
    "dialect":          dialects,
    "predicted_label":  y_pred,
    "predicted_class":  ["Native" if p == 1 else "Non-Native" for p in y_pred],
    "confidence_score": np.round(confidence, 4),
    "prob_native":      np.round(prob_native, 4),
    "prob_non_native":  np.round(prob_non_native, 4),
    "true_label":       y_test,
    "true_class":       ["Native" if t == 1 else "Non-Native" for t in y_test],
    "correct":          (y_pred == y_test),
})

predictions_path = f"{PROJECT_ROOT}/outputs/predictions.csv"
results.to_csv(predictions_path, index=False)
n_correct = results["correct"].sum()
n_total   = len(results)
print(f"predictions.csv saved ({n_total} rows, {n_correct}/{n_total} correct)")

# ── Load LSTM config for report ───────────────────────────────────
try:
    lstm_config = joblib.load(f"{PROJECT_ROOT}/outputs/lstm_config.joblib")
    lstm_params_str = (
        f"input_dim={lstm_config['input_dim']}, "
        f"hidden_dim={lstm_config['hidden_dim']}, "
        f"num_layers={lstm_config['num_layers']}, "
        f"bidirectional=True"
    )
except Exception:
    lstm_params_str = "See stage8 logs"

# ── technical_report.md ───────────────────────────────────────────
data    = np.load(f"{PROJECT_ROOT}/outputs/stage7_features.npz",
                   allow_pickle=True)
n_train = int(data["X_train"].shape[0])
n_test  = int(len(y_test))
today   = date.today().strftime("%B %d, %Y")

report = f"""# Technical Evaluation Report
## Team Databaes — Arabic Native/Non-Native Speech Classification
**Client:** Renan Partners Private Limited
**Date:** {today}

---

## 1. Executive Summary

A binary classifier was developed to distinguish Native from Non-Native Arabic speakers.
The model combines deep acoustic representations from OpenAI's Whisper large-v3 encoder with
hand-crafted prosodic features. A bidirectional LSTM encodes recording-level temporal
patterns from variable-length segment sequences, and a Random Forest classifier produces
the final decision.

**Final Results on Renan Test Set (n={n_test} recording sequences):**

| Metric | Value |
|--------|-------|
| Accuracy | {metrics['accuracy']*100:.2f}% |
| F1 Score | {metrics['f1_score']:.4f} |
| Precision | {metrics['precision']:.4f} |
| Recall | {metrics['recall']:.4f} |
| ROC AUC | {metrics['roc_auc']:.4f} |
| EER | {metrics['eer']*100:.2f}% |
| 95% CI (Accuracy) | [{metrics['ci_low']*100:.1f}%, {metrics['ci_high']*100:.1f}%] |

---

## 2. Methodology

### 2.1 Data
- **Training source:** 160 Renan recordings (Gulf dialects: SA/QA/AE/KW) +
  200 Mozilla Common Voice 24.0 Arabic clips (native reference)
- **Augmentation:** Time stretching (±10%), pitch shifting (±2 semitones), Gaussian noise —
  training segments only. Non-native (minority) received 5 augmentations; native received 1.

### 2.2 Feature Extraction

**Stream 1 — Whisper Acoustic Embeddings (1280-dim)**
- Model: `openai/whisper-large-v3` (Radford et al., 2023)
- Used frozen encoder only — no fine-tuning (prevents overfitting on 160 training recordings)
- Whisper converts audio to 80-bin log-Mel spectrograms internally
- Each 3-second segment → encoder hidden states → mean-pooled → 1280-dimensional vector
- Whisper's multilingual pretraining on 680k hours of audio provides rich acoustic
  representations that capture phonetic, prosodic, and speaker characteristics

**Stream 2 — Prosodic Features (6-dim)**

| Feature | Description |
|---------|-------------|
| F0 mean | Average fundamental frequency |
| F0 variance | Pitch dynamics |
| Speaking rate | Syllable-proxy peaks per second |
| Pause frequency | Pauses (>200ms) per second |
| Mean pause duration | Average pause length in seconds |
| nPVI | Normalised Pairwise Variability Index (rhythmic stress-timing) |

### 2.3 Fusion and Normalisation
- Streams concatenated → 1286-dimensional vector per segment
- StandardScaler fitted on training data only
- Segments grouped by recording into temporal sequences

### 2.4 Classifier

**Stage 1 — LSTM Temporal Encoder**
- Architecture: Bidirectional LSTM ({lstm_params_str})
- Processes variable-length sequences of 1286-dim segment features per recording
- Outputs a fixed 256-dim recording-level representation
- Trained end-to-end with BCE loss (class-balanced via pos_weight) + early stopping

**Stage 2 — Random Forest**
- 500 trees, `class_weight='balanced'`
- Trained on LSTM-extracted 256-dim hidden-state vectors
- Provides calibrated probability estimates via class vote fractions

---

## 3. Results

### 3.1 Confusion Matrix

|  | Predicted Non-Native | Predicted Native |
|--|--|--|
| **True Non-Native** | {int(metrics['tn'])} | {int(metrics['fp'])} |
| **True Native** | {int(metrics['fn'])} | {int(metrics['tp'])} |

- **False Positives** (non-native classified as native): {int(metrics['fp'])}
- **False Negatives** (native classified as non-native): {int(metrics['fn'])}

### 3.2 Statistical Caveat
The 95% CI [{metrics['ci_low']*100:.1f}%, {metrics['ci_high']*100:.1f}%] is relatively wide due to the
small test set (n={n_test}). Results would be more stable with n≥100 test samples.

---

## 4. Known Limitations

1. **Small test set** — wide confidence intervals
2. **Kuwaiti dialect** — only ~1.9% of training data (3 files); may generalise poorly to KW test samples
3. **Short Common Voice clips** — 3–10s clips yield less reliable prosodic estimates
4. **Non-native L1 diversity** — model trained primarily on Gulf-native speakers

---

## 5. References

- Radford, A. et al. (2023). Robust Speech Recognition via Large-Scale Weak Supervision.
  https://arxiv.org/abs/2212.04356
- Hochreiter & Schmidhuber (1997). Long Short-Term Memory. Neural Computation, 9(8).
- Breiman (2001). Random Forests. Machine Learning, 45(1).
- Emara & Shaker (2024). Speech Communication, 157.
- Grabe & Low (2002). Durational variability and the Rhythm Class Hypothesis.
- Mozilla Common Voice 24.0 Arabic, CC0-1.0.
- OpenAI Whisper. https://github.com/openai/whisper

---
*Report generated automatically by Stage 10 — Team Databaes*
"""

report_path = f"{PROJECT_ROOT}/outputs/technical_report.md"
with open(report_path, "w") as f:
    f.write(report)

print("technical_report.md saved")
print("\n" + "="*50)
print("  ALL STAGE 10 DELIVERABLES COMPLETE")
print("="*50)
print(f"  {predictions_path}")
print(f"  {report_path}")
print("="*50)